# Baseline steganalysis: KV residual Xu-Net

This notebook builds a small, reproducible baseline for detecting the stego tiles in `raise512-run1`. It keeps clean and stego copies of each cover together, splits by source photo, and uses a fixed 5×5 KV residual filter before an Xu-Net-style classifier.

Set `QUICK = True` in the setup cell for a short GPU smoke test. Set it to `False` for the full run; the notebook does not start the full run automatically.

This v2 version avoids the train/validation gap seen in notebook 01: training uses unpaired clean and stego images by default, selects the best checkpoint by validation accuracy, and uses a small AdamW weight decay. Validation and test remain pair-based and run in evaluation mode.


## 1. Setup

This cell defines paths, seeds, the run mode, and the device. The expected result is a CUDA device on the ROCm container (PyTorch calls the AMD GPU `cuda`), or CPU when no accelerator is available.


In [ ]:
from pathlib import Path
import json
import os
import random
import time

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, __version__ as PIL_VERSION
from torch.utils.data import DataLoader, Dataset

root_candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((candidate for candidate in root_candidates if (candidate / "Datasets").is_dir()), None)
if ROOT is None and (Path("/work") / "Datasets").is_dir():
    ROOT = Path("/work")
if ROOT is None:
    ROOT = Path.cwd()
DATA_ROOT = ROOT / "Datasets" / "generated" / "raise512-run1-records"
LABELS_PATH = DATA_ROOT / "labels.csv"
SPLIT_PATH = DATA_ROOT / "splits" / "split_seed0.csv"
CHECKPOINT_ROOT = ROOT / "runs"

SEED = 0
QUICK = True
PAIRED_BATCHES = False
RUN_NAME = "baseline_kv_xunet_v2_quick" if QUICK else "baseline_kv_xunet_v2"
EPOCHS = 2 if QUICK else 10
# Full training uses 32 images per step, the same as notebook 01.
BATCH_SIZE = 8 if QUICK else 32
QUICK_PAIRS = {"train": 32, "val": 16, "test": 16}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
L.seed_everything(SEED, workers=True)

assert LABELS_PATH.exists(), f"Missing labels file: {LABELS_PATH}"
assert DATA_ROOT.exists(), f"Missing dataset directory: {DATA_ROOT}"
print(f"Root: {ROOT}")
print(f"Dataset: {DATA_ROOT}")
print(f"QUICK={QUICK}, run={RUN_NAME}, epochs={EPOCHS}, batch_size={BATCH_SIZE} images, paired_batches={PAIRED_BATCHES}")
print(f"Device: {DEVICE}; torch.cuda.is_available()={torch.cuda.is_available()}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


The core packages are part of the project environment. `pandas` and `matplotlib` make the tables and figures convenient, but they are not listed in `pyproject.toml` yet; install them separately if needed.


In [ ]:
print("torch", torch.__version__)
print("lightning", L.__version__)
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("Pillow", PIL_VERSION)
print("matplotlib", plt.matplotlib.__version__)


## 2. Load labels and inspect settings

This cell loads one row per clean or stego image. The expected result is 99,592 rows, balanced between labels 0 and 1, with counts printed for camera, `k`, bit mode, method, and placement.


In [ ]:
labels = pd.read_csv(LABELS_PATH)
print(f"labels shape: {labels.shape}")
print("\nCounts by label:")
print(labels["label"].value_counts().sort_index().to_string())
for column in ["camera_model", "k", "bit_mode", "method", "placement"]:
    print(f"\nCounts by {column}:")
    print(labels[column].value_counts(dropna=False).sort_index().to_string())


## 3. Make or reuse a source-photo split

A clean tile and its stego copy must stay together, and tiles from one RAISE source photo must not leak across splits. This cell makes a fixed-seed 70/15/15 split of source photos, stratified by camera model, and writes only `file,split` to `split_seed0.csv` when that file is absent. The expected result is a reuse message on later runs and an assertion that source-photo sets do not overlap.


In [ ]:
def make_pairs(frame: pd.DataFrame) -> pd.DataFrame:
    """Merge each clean tile with its matching stego tile."""
    clean = frame.loc[frame["label"] == 0].copy().rename(columns={"file": "file_clean"})
    stego = frame.loc[frame["label"] == 1].copy().rename(columns={"file": "file_stego"})
    pairs = clean.merge(stego, on="cover_id", suffixes=("_clean", "_stego"), validate="one_to_one")
    for column in ["source_photo", "camera_model", "tile_row", "tile_col"]:
        left = pairs[f"{column}_clean"].astype(str)
        right = pairs[f"{column}_stego"].astype(str)
        assert left.equals(right), f"clean/stego mismatch in {column}"
        pairs[column] = pairs[f"{column}_stego"]
    for column in ["k", "bit_mode", "method", "placement", "payload_kind", "bpp_actual"]:
        pairs[column] = pairs[f"{column}_stego"]
    keep = [
        "cover_id", "file_clean", "file_stego", "source_photo", "camera_model",
        "tile_row", "tile_col", "k", "bit_mode", "method", "placement",
        "payload_kind", "bpp_actual",
    ]
    return pairs[keep].reset_index(drop=True)


def source_split(pair_frame: pd.DataFrame, path: Path, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create or reuse a camera-stratified file-to-split map."""
    file_info = pd.concat([
        pair_frame[["file_clean", "source_photo", "camera_model"]].rename(columns={"file_clean": "file"}),
        pair_frame[["file_stego", "source_photo", "camera_model"]].rename(columns={"file_stego": "file"}),
    ], ignore_index=True)
    assert not file_info["file"].duplicated().any()
    source_info = file_info[["source_photo", "camera_model"]].drop_duplicates()
    assert not source_info["source_photo"].duplicated().any()
    if path.exists():
        saved = pd.read_csv(path, dtype={"file": str, "split": str})
        print(f"Reusing existing split: {path}")
    else:
        rng = np.random.default_rng(seed)
        source_rows = []
        for camera, group in source_info.groupby("camera_model", sort=True):
            sources = group["source_photo"].to_numpy(copy=True)
            rng.shuffle(sources)
            n = len(sources)
            n_train = int(round(n * 0.70))
            n_val = int(round(n * 0.15))
            n_test = n - n_train - n_val
            assert n_test >= 0
            for split_name, values in zip(
                ["train", "val", "test"],
                [sources[:n_train], sources[n_train:n_train + n_val], sources[n_train + n_val:]],
            ):
                source_rows.extend({"source_photo": str(source), "split": split_name} for source in values)
        source_assignments = pd.DataFrame(source_rows, columns=["source_photo", "split"])
        saved = file_info[["file", "source_photo"]].merge(source_assignments, on="source_photo", validate="many_to_one")[
            ["file", "split"]
        ]
        path.parent.mkdir(parents=True, exist_ok=True)
        saved.to_csv(path, index=False)
        print(f"Wrote new split: {path}")
    assert list(saved.columns) == ["file", "split"]
    assert set(saved["file"]) == set(file_info["file"])
    assert not saved["file"].duplicated().any()
    assert set(saved["split"]) <= {"train", "val", "test"}
    file_with_source = file_info.merge(saved, on="file", validate="one_to_one")
    source_assignments = file_with_source[["source_photo", "split"]].drop_duplicates()
    assert not source_assignments["source_photo"].duplicated().any()
    assert set(source_assignments["source_photo"]) == set(source_info["source_photo"])
    return saved, source_assignments

pairs = make_pairs(labels)
print(f"Paired covers: {len(pairs):,}; expected 49,796")
split, source_assignments = source_split(pairs, SPLIT_PATH, SEED)
pair_splits = pairs.merge(source_assignments, on="source_photo", how="left", validate="many_to_one")
assert not pair_splits["split"].isna().any()
source_sets = {name: set(source_assignments.loc[source_assignments["split"] == name, "source_photo"]) for name in ["train", "val", "test"]}
assert not (source_sets["train"] & source_sets["val"])
assert not (source_sets["train"] & source_sets["test"])
assert not (source_sets["val"] & source_sets["test"])
print("Source-photo overlap check: passed")
print("Source counts by camera and split:")
print(source_assignments.merge(pairs[["source_photo", "camera_model"]].drop_duplicates(), on="source_photo").groupby(["camera_model", "split"]).size().unstack(fill_value=0).to_string())
print("Pair counts by split:")
print(pair_splits["split"].value_counts().sort_index().to_string())

if QUICK:
    run_frames = []
    for offset, split_name in enumerate(["train", "val", "test"]):
        available = pair_splits.loc[pair_splits["split"] == split_name]
        n = min(QUICK_PAIRS[split_name], len(available))
        run_frames.append(available.sample(n=n, random_state=SEED + offset).copy())
    run_pairs = pd.concat(run_frames, ignore_index=True)
else:
    run_pairs = pair_splits.copy()
print("Pairs used in this run:")
print(run_pairs["split"].value_counts().sort_index().to_string())


## 4. Inspect one clean/stego pair

This cell displays a clean tile, its stego copy, their absolute pixel difference, and the absolute KV residual. The expected result is a tiny sparse difference for the embedded payload and a residual view that emphasizes high-frequency content.


In [ ]:
KV_ARRAY = np.array([
    [-1, 2, -2, 2, -1],
    [2, -6, 8, -6, 2],
    [-2, 8, -12, 8, -2],
    [2, -6, 8, -6, 2],
    [-1, 2, -2, 2, -1],
], dtype=np.float32) / 12.0


def read_raw(path: Path) -> torch.Tensor:
    """Read an RGB PNG as CHW float32 values in the original 0-255 range."""
    with Image.open(path) as image:
        array = np.asarray(image.convert("RGB"), dtype=np.uint8).copy()
    return torch.from_numpy(array).permute(2, 0, 1).contiguous().float()


def kv_residual(image: torch.Tensor) -> torch.Tensor:
    """Apply the fixed KV filter independently to R, G, and B."""
    kernel = torch.from_numpy(KV_ARRAY).to(dtype=image.dtype, device=image.device)
    weight = kernel.view(1, 1, 5, 5).repeat(3, 1, 1, 1)
    return F.conv2d(image.unsqueeze(0), weight, padding=2, groups=3).squeeze(0)

example = run_pairs.iloc[0]
clean_example = read_raw(DATA_ROOT / example["file_clean"])
stego_example = read_raw(DATA_ROOT / example["file_stego"])
difference = (stego_example - clean_example).abs().numpy().mean(axis=0)
residual = kv_residual(clean_example).abs().numpy().mean(axis=0)
residual_limit = max(float(np.percentile(residual, 99.5)), 1.0)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow((clean_example.permute(1, 2, 0).numpy() / 255.0).clip(0, 1)); axes[0].set_title("clean")
axes[1].imshow((stego_example.permute(1, 2, 0).numpy() / 255.0).clip(0, 1)); axes[1].set_title("stego")
axes[2].imshow(difference, cmap="magma", vmin=0, vmax=max(float(difference.max()), 1.0)); axes[2].set_title("absolute difference")
axes[3].imshow(residual, cmap="gray", vmin=0, vmax=residual_limit); axes[3].set_title("|KV residual|")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()
print(f"Pair: {example['file_clean']} / {example['file_stego']}")
print(f"Shapes: clean={tuple(clean_example.shape)}, stego={tuple(stego_example.shape)}")
print(f"Mean absolute pixel difference: {difference.mean():.5f}; nonzero pixels: {(difference > 0).sum():,}")
print(f"KV residual range: {residual.min():.3f} to {residual.max():.3f}")


## 5. Dataset and DataLoader

Validation and test use clean/stego pairs so their predictions and per-setting metrics stay aligned. With `PAIRED_BATCHES = False`, training uses a separate sample for each image and shuffles those samples independently. Inputs remain raw float32 values from 0 to 255, with no resize or normalization.


In [ ]:
class PairDataset(Dataset):
    """Load paired tiles for validation, testing, and optional paired training."""

    def __init__(self, frame: pd.DataFrame, train: bool = False):
        self.frame = frame.reset_index(drop=True)
        self.train = train

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        clean = read_raw(DATA_ROOT / row["file_clean"])
        stego = read_raw(DATA_ROOT / row["file_stego"])
        if self.train:
            clean = augment_image(clean)
            stego = augment_image(stego)
        return clean, stego, index


def augment_image(image: torch.Tensor) -> torch.Tensor:
    """Apply the baseline's random geometric augmentation to one image."""
    if bool(torch.rand(()) < 0.5):
        image = torch.flip(image, dims=(-1,))
    if bool(torch.rand(()) < 0.5):
        image = torch.flip(image, dims=(-2,))
    turns = int(torch.randint(0, 4, ()).item())
    return torch.rot90(image, turns, dims=(-2, -1))


class ImageDataset(Dataset):
    """Expose clean and stego tiles as separate shuffled training samples."""

    def __init__(self, frame: pd.DataFrame, train: bool = False):
        self.train = train
        self.samples = []
        for row in frame.itertuples(index=False):
            self.samples.append((row.file_clean, 0))
            self.samples.append((row.file_stego, 1))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        file_name, label = self.samples[index]
        image = read_raw(DATA_ROOT / file_name)
        if self.train:
            image = augment_image(image)
        return image, label, index


train_frame = run_pairs.loc[run_pairs["split"] == "train"].reset_index(drop=True)
val_frame = run_pairs.loc[run_pairs["split"] == "val"].reset_index(drop=True)
test_frame = run_pairs.loc[run_pairs["split"] == "test"].reset_index(drop=True)
train_dataset = PairDataset(train_frame, train=True) if PAIRED_BATCHES else ImageDataset(train_frame, train=True)
val_dataset = PairDataset(val_frame, train=False)
test_dataset = PairDataset(test_frame, train=False)
NUM_WORKERS = min(8, max(2, (os.cpu_count() or 4) - 2))
if QUICK:
    NUM_WORKERS = min(2, NUM_WORKERS)
loader_kwargs = {"num_workers": NUM_WORKERS, "pin_memory": DEVICE.type == "cuda"}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
pair_batch_size = max(1, BATCH_SIZE // 2)
train_batch_size = pair_batch_size if PAIRED_BATCHES else BATCH_SIZE
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=pair_batch_size, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=pair_batch_size, shuffle=False, **loader_kwargs)
train_eval_frame = train_frame.sample(n=min(1000, len(train_frame)), random_state=SEED).reset_index(drop=True)
train_eval_dataset = PairDataset(train_eval_frame, train=False)
train_eval_loader = DataLoader(train_eval_dataset, batch_size=pair_batch_size, shuffle=False, **loader_kwargs)

preview_clean, preview_stego, _ = PairDataset(train_frame.head(1), train=False)[0]
clean_batch = preview_clean.unsqueeze(0)
stego_batch = preview_stego.unsqueeze(0)
if PAIRED_BATCHES:
    batch_clean, batch_stego, pair_indices = next(iter(train_loader))
    print(f"paired training batch: clean={tuple(batch_clean.shape)}, stego={tuple(batch_stego.shape)}")
else:
    image_batch, target_batch, image_indices = next(iter(train_loader))
    print(f"unpaired training batch: images={tuple(image_batch.shape)}, targets={tuple(target_batch.shape)}")
print(f"num_workers={NUM_WORKERS}; train batches={len(train_loader)}, val batches={len(val_loader)}, test batches={len(test_loader)}")
print(f"preview clean={tuple(clean_batch.shape)}, preview stego={tuple(stego_batch.shape)}, dtype={clean_batch.dtype}")
print(f"range: {clean_batch.min().item():.1f} to {clean_batch.max().item():.1f}")
assert clean_batch.shape == stego_batch.shape
assert clean_batch.dtype == torch.float32
assert 0 <= clean_batch.min() and clean_batch.max() <= 255


## 6. Xu-Net-style residual model

The model first applies a frozen 5×5 KV filter independently to each color channel. It then follows the requested convolution, absolute-value, batch-normalization, activation, average-pooling, and global-average-pooling sequence. The expected result is a parameter count and a two-class output shape.

In [ ]:
KV_TENSOR = torch.tensor(KV_ARRAY, dtype=torch.float32)


class KVFilter(nn.Module):
    """Frozen depthwise KV residual filter."""

    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 3, kernel_size=5, padding=2, groups=3, bias=False)
        with torch.no_grad():
            self.conv.weight.copy_(KV_TENSOR.view(1, 1, 5, 5).repeat(3, 1, 1, 1))
        self.conv.weight.requires_grad_(False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class XuNet(nn.Module):
    """Compact Xu-Net-style classifier for 512x512 RGB tiles."""

    def __init__(self):
        super().__init__()
        self.kv = KVFilter()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 64, kernel_size=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=1)
        self.bn5 = nn.BatchNorm2d(128)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
        self.classifier = nn.Linear(128, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.kv(x)
        x = self.pool(torch.tanh(self.bn1(torch.abs(self.conv1(x)))))
        x = self.pool(torch.tanh(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x))))
        x = x.mean(dim=(-2, -1))
        return self.classifier(x)


model_for_count = XuNet()
total_parameters = sum(parameter.numel() for parameter in model_for_count.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model_for_count.parameters() if parameter.requires_grad)
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Output shape for one batch: {tuple(model_for_count(clean_batch[:1]).shape)}")


The KV weights must remain fixed during optimization. This cell runs one real optimizer step on a batch and asserts that the frozen filter is unchanged. The expected result is `KV weights unchanged: True`.

In [ ]:
probe = XuNet().to(DEVICE)
probe.train()
probe_optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3)
probe_clean = clean_batch.to(DEVICE)
probe_stego = stego_batch.to(DEVICE)
probe_images = torch.cat([probe_clean, probe_stego], dim=0)
probe_targets = torch.cat([
    torch.zeros(probe_clean.shape[0], dtype=torch.long, device=DEVICE),
    torch.ones(probe_stego.shape[0], dtype=torch.long, device=DEVICE),
])
kv_before = probe.kv.conv.weight.detach().clone()
probe_optimizer.zero_grad(set_to_none=True)
probe_loss = F.cross_entropy(probe(probe_images), probe_targets)
probe_loss.backward()
probe_optimizer.step()
kv_unchanged = torch.equal(kv_before, probe.kv.conv.weight.detach())
print(f"Probe loss: {probe_loss.item():.5f}")
print(f"KV weights unchanged: {kv_unchanged}")
assert kv_unchanged


## 7. Train with Lightning

This cell trains with cross-entropy, AdamW, and a step decay. By default, training yields clean and stego tiles as separate samples and shuffles them independently, so a batch does not provide a clean/stego shortcut through BatchNorm. Set `PAIRED_BATCHES = True` only to reproduce the old paired behavior. Validation remains pair-based. The best checkpoint is selected by validation accuracy, and `last.ckpt` is saved every epoch.

**Precision note.** Keep fp32 and ignore Lightning's `set_float32_matmul_precision` tip: the KV residual needs full precision because ±1 pixel changes become ±1/12 after KV on values up to 255. Channels-last and `cudnn.benchmark` were measured but gave no useful real-data speed-up on this GPU, so v2 uses `deterministic=False`. The PC speed test selected default MIOpen settings; `MIOPEN_FIND_MODE=1` gave no gain.


In [ ]:
class SteganalysisModule(L.LightningModule):
    """Lightning wrapper for paired or unpaired image batches."""

    def __init__(self, lr: float = 1e-3, max_epochs: int = 2, weight_decay: float = 1e-4, paired_batches: bool = False):
        super().__init__()
        self.save_hyperparameters()
        self.net = XuNet()

    def pair_logits_targets(self, batch):
        clean, stego, _ = batch
        images = torch.cat([clean, stego], dim=0)
        targets = torch.cat([
            torch.zeros(clean.shape[0], dtype=torch.long, device=clean.device),
            torch.ones(stego.shape[0], dtype=torch.long, device=stego.device),
        ])
        return self.net(images), targets

    def training_logits_targets(self, batch):
        if self.hparams.paired_batches:
            return self.pair_logits_targets(batch)
        images, targets, _ = batch
        return self.net(images), targets

    def training_step(self, batch, batch_idx):
        logits, targets = self.training_logits_targets(batch)
        loss = F.cross_entropy(logits, targets)
        accuracy = (logits.argmax(dim=1) == targets).float().mean()
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=targets.numel())
        self.log("train_acc", accuracy, on_step=False, on_epoch=True, prog_bar=True, batch_size=targets.numel())
        return loss

    def validation_step(self, batch, batch_idx):
        logits, targets = self.pair_logits_targets(batch)
        loss = F.cross_entropy(logits, targets)
        accuracy = (logits.argmax(dim=1) == targets).float().mean()
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=targets.numel())
        self.log("val_acc", accuracy, on_step=False, on_epoch=True, prog_bar=True, batch_size=targets.numel())

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer, step_size=max(1, self.hparams.max_epochs // 2), gamma=0.1
        )
        return {"optimizer": optimizer, "lr_scheduler": scheduler}


checkpoint_dir = CHECKPOINT_ROOT / RUN_NAME
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_callback = L.pytorch.callbacks.ModelCheckpoint(
    dirpath=str(checkpoint_dir),
    filename="epoch{epoch:02d}-val_acc{val_acc:.4f}",
    auto_insert_metric_name=False,
    save_last=True,
    save_top_k=1,
    monitor="val_acc",
    mode="max",
)
trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu" if DEVICE.type == "cuda" else "cpu",
    devices=1,
    callbacks=[checkpoint_callback],
    default_root_dir=str(CHECKPOINT_ROOT),
    log_every_n_steps=1,
    deterministic=False,
)
training_started = time.perf_counter()
lightning_model = SteganalysisModule(max_epochs=EPOCHS, paired_batches=PAIRED_BATCHES)
trainer.fit(lightning_model, train_loader, val_loader, ckpt_path="last")
training_seconds = time.perf_counter() - training_started
best_checkpoint = checkpoint_callback.best_model_path or checkpoint_callback.last_model_path
print(f"Training time: {training_seconds / 60:.2f} minutes")
print(f"Last checkpoint: {checkpoint_callback.last_model_path}")
print(f"Best checkpoint: {best_checkpoint}")
print(f"Trainer device: {trainer.strategy.root_device}")


## 8. Evaluate the best checkpoint

This cell loads the best validation-accuracy checkpoint and predicts both members of the train sample, validation, and test pairs. Predictions use evaluation mode with running BatchNorm statistics. It reports train accuracy beside validation and test accuracy as a gap check, then computes AUC and equal-error-style `P_E` for the held-out test tiles.


In [ ]:
def _threshold_counts(labels_array: np.ndarray, scores_array: np.ndarray) -> tuple[np.ndarray, np.ndarray, int, int]:
    """Count positives and negatives at tied score thresholds."""
    labels_array = np.asarray(labels_array, dtype=np.int64)
    scores_array = np.asarray(scores_array, dtype=np.float64)
    if labels_array.size == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.int64), 0, 0
    order = np.argsort(-scores_array, kind="mergesort")
    sorted_scores = scores_array[order]
    sorted_labels = labels_array[order]
    group_ends = np.flatnonzero(sorted_scores[1:] != sorted_scores[:-1]) + 1
    group_ends = np.r_[group_ends, len(sorted_scores)]
    cumulative_positive = np.cumsum(sorted_labels == 1, dtype=np.int64)
    cumulative_negative = np.cumsum(sorted_labels == 0, dtype=np.int64)
    return (
        cumulative_positive[group_ends - 1],
        cumulative_negative[group_ends - 1],
        int((labels_array == 1).sum()),
        int((labels_array == 0).sum()),
    )


def auc_numpy(labels_array: np.ndarray, scores_array: np.ndarray) -> float:
    """Compute ROC AUC with one sort and tied-score groups."""
    true_positive, false_positive, positives, negatives = _threshold_counts(labels_array, scores_array)
    if positives == 0 or negatives == 0:
        return 0.0
    denominator_positive = float(positives)
    denominator_negative = float(negatives)
    true_positive_rate = np.r_[0.0, true_positive / denominator_positive, true_positive[-1] / denominator_positive]
    false_positive_rate = np.r_[0.0, false_positive / denominator_negative, false_positive[-1] / denominator_negative]
    return float(np.trapezoid(true_positive_rate, false_positive_rate))


def probability_error(labels_array: np.ndarray, scores_array: np.ndarray) -> float:
    """Compute minimum P_E with one sort and tied-score groups."""
    true_positive, false_positive, positives, negatives = _threshold_counts(labels_array, scores_array)
    if positives == 0 and negatives == 0:
        return 0.0
    if negatives:
        false_alarm = np.r_[0.0, false_positive / negatives, false_positive[-1] / negatives]
    else:
        false_alarm = np.zeros(len(true_positive) + 2, dtype=np.float64)
    if positives:
        missed_detection = np.r_[1.0, (positives - true_positive) / positives, 0.0]
    else:
        missed_detection = np.zeros(len(true_positive) + 2, dtype=np.float64)
    return float(np.min((false_alarm + missed_detection) / 2.0))


def predict_pairs(model: SteganalysisModule, loader: DataLoader) -> pd.DataFrame:
    """Predict clean and stego probabilities while retaining pair metadata."""
    model.eval()
    frame = loader.dataset.frame
    rows = []
    with torch.inference_mode():
        for clean, stego, indices in loader:
            clean = clean.to(DEVICE, non_blocking=True)
            stego = stego.to(DEVICE, non_blocking=True)
            probabilities = torch.softmax(model.net(torch.cat([clean, stego], dim=0)), dim=1)[:, 1]
            count = clean.shape[0]
            clean_probabilities = probabilities[:count].cpu().numpy()
            stego_probabilities = probabilities[count:].cpu().numpy()
            for position, pair_index in enumerate(indices.tolist()):
                row = frame.iloc[int(pair_index)]
                common = {
                    "cover_id": row["cover_id"],
                    "source_photo": row["source_photo"],
                    "camera_model": row["camera_model"],
                    "k": row["k"],
                    "bit_mode": row["bit_mode"],
                    "method": row["method"],
                    "placement": row["placement"],
                    "payload_kind": row["payload_kind"],
                    "bpp_actual": row["bpp_actual"],
                }
                rows.append({**common, "file": row["file_clean"], "label": 0, "prob_stego": float(clean_probabilities[position])})
                rows.append({**common, "file": row["file_stego"], "label": 1, "prob_stego": float(stego_probabilities[position])})
    return pd.DataFrame(rows)


best_model = SteganalysisModule.load_from_checkpoint(best_checkpoint, map_location=DEVICE)
best_model.to(DEVICE)
train_predictions = predict_pairs(best_model, train_eval_loader)
val_predictions = predict_pairs(best_model, val_loader)
predictions = predict_pairs(best_model, test_loader)

def prediction_accuracy(frame: pd.DataFrame) -> float:
    y_true = frame["label"].to_numpy(dtype=np.int64)
    y_score = frame["prob_stego"].to_numpy(dtype=np.float64)
    return float(((y_score >= 0.5).astype(np.int64) == y_true).mean())

print(f"Train eval images: {len(train_predictions):,}; accuracy: {prediction_accuracy(train_predictions):.4f}")
print(f"Validation eval images: {len(val_predictions):,}; accuracy: {prediction_accuracy(val_predictions):.4f}")
print(f"Test eval images: {len(predictions):,}; accuracy: {prediction_accuracy(predictions):.4f}")
y_true = predictions["label"].to_numpy(dtype=np.int64)
y_score = predictions["prob_stego"].to_numpy(dtype=np.float64)
y_hat = (y_score >= 0.5).astype(np.int64)
accuracy = float((y_hat == y_true).mean())
auc = auc_numpy(y_true, y_score)
p_e = probability_error(y_true, y_score)
print(f"Test images: {len(predictions):,}")
print(f"Accuracy: {accuracy:.4f}")
print(f"AUC: {auc:.4f}")
print(f"P_E: {p_e:.4f}")


### Metric implementation check

This small check compares the fast cumulative-count implementations with the original threshold loops on 500 scores containing ties. The expected result is two `True` lines within `1e-12`; tied scores must move together.

In [ ]:
def reference_auc(labels_array: np.ndarray, scores_array: np.ndarray) -> float:
    thresholds = np.r_[np.inf, np.sort(np.unique(scores_array))[::-1], -np.inf]
    positives = max(int((labels_array == 1).sum()), 1)
    negatives = max(int((labels_array == 0).sum()), 1)
    tpr = []
    fpr = []
    for threshold in thresholds:
        predicted = scores_array >= threshold
        tpr.append(float(((predicted == 1) & (labels_array == 1)).sum()) / positives)
        fpr.append(float(((predicted == 1) & (labels_array == 0)).sum()) / negatives)
    return float(np.trapezoid(np.asarray(tpr), np.asarray(fpr)))


def reference_probability_error(labels_array: np.ndarray, scores_array: np.ndarray) -> float:
    positives = labels_array == 1
    negatives = labels_array == 0
    thresholds = np.r_[np.inf, np.sort(np.unique(scores_array))[::-1], -np.inf]
    errors = []
    for threshold in thresholds:
        predicted = scores_array >= threshold
        p_fa = float(predicted[negatives].mean()) if negatives.any() else 0.0
        p_md = float((~predicted[positives]).mean()) if positives.any() else 0.0
        errors.append((p_fa + p_md) / 2.0)
    return float(min(errors))

check_rng = np.random.default_rng(123)
check_labels = check_rng.integers(0, 2, size=500, dtype=np.int64)
check_scores = np.round(check_rng.random(500), 2)
assert np.unique(check_scores).size < len(check_scores), "test array must contain ties"
fast_auc = auc_numpy(check_labels, check_scores)
reference_auc_value = reference_auc(check_labels, check_scores)
fast_p_e = probability_error(check_labels, check_scores)
reference_p_e = reference_probability_error(check_labels, check_scores)
assert np.isclose(fast_auc, reference_auc_value, atol=1e-12, rtol=0)
assert np.isclose(fast_p_e, reference_p_e, atol=1e-12, rtol=0)
print(f"AUC agrees within 1e-12: {np.isclose(fast_auc, reference_auc_value, atol=1e-12, rtol=0)}")
print(f"P_E agrees within 1e-12: {np.isclose(fast_p_e, reference_p_e, atol=1e-12, rtol=0)}")


## 9. Per-setting `P_E`

For every stego setting, this cell adds its paired clean image and recomputes `P_E`. It reports rows by combined `bit_mode/k`, bit mode, method, placement, method × placement, payload kind, and the requested bpp bins. The expected result is a table with pair counts and a `P_E` value for each setting present in the test subset.

In [ ]:
stego_predictions = predictions.loc[predictions["label"] == 1].copy()
clean_predictions = predictions.loc[predictions["label"] == 0].copy()
stego_predictions["bit_mode_k"] = (
    stego_predictions["bit_mode"].astype(str) + "/" + pd.to_numeric(stego_predictions["k"], errors="raise").astype(int).astype(str)
)
stego_predictions["method_x_placement"] = (
    stego_predictions["method"].astype(str) + "/" + stego_predictions["placement"].astype(str)
)
stego_predictions["bpp_bin"] = pd.cut(
    pd.to_numeric(stego_predictions["bpp_actual"], errors="coerce"),
    bins=[0.05, 0.15, 0.30, 0.50],
    labels=["0.05-0.15", "0.15-0.30", "0.30-0.50"],
    include_lowest=True,
    right=True,
)

setting_rows = []
setting_specs = [
    ("bit_mode/k", "bit_mode_k"),
    ("bit_mode", "bit_mode"),
    ("method", "method"),
    ("placement", "placement"),
    ("method x placement", "method_x_placement"),
    ("payload_kind", "payload_kind"),
    ("bpp_bin", "bpp_bin"),
]
for setting_label, setting_column in setting_specs:
    for value, group in stego_predictions.groupby(setting_column, dropna=False, observed=True):
        if pd.isna(value):
            continue
        cover_ids = set(group["cover_id"])
        paired_clean = clean_predictions.loc[clean_predictions["cover_id"].isin(cover_ids)]
        paired_images = pd.concat([paired_clean, group], ignore_index=True)
        group_labels = paired_images["label"].to_numpy(dtype=np.int64)
        group_scores = paired_images["prob_stego"].to_numpy(dtype=np.float64)
        setting_rows.append({
            "setting": setting_label,
            "value": str(value),
            "pairs": len(group),
            "P_E": probability_error(group_labels, group_scores),
        })
per_setting = pd.DataFrame(setting_rows, columns=["setting", "value", "pairs", "P_E"])
print(per_setting.to_string(index=False))


## 10. Save predictions

The final cell saves one row per test image under the run checkpoint folder, including the file, label, predicted stego probability, source photo, and setting metadata. The expected result is a CSV that can reproduce the aggregate and per-setting metrics.

In [ ]:
predictions_path = checkpoint_dir / "test_predictions.csv"
predictions.to_csv(predictions_path, index=False)
print(f"Saved {len(predictions):,} predictions to {predictions_path}")
print(f"Run folder contents: {[path.name for path in sorted(checkpoint_dir.iterdir())]}")
